# Governed Response Composer Tutorial

This tutorial demonstrates SIM-ONE's self-governing AI response system that automatically generates and refines responses until they meet governance standards.

## What You'll Learn
- Generate AI responses that automatically consult SIM-ONE Framework documentation
- Validate responses against the Five Laws of Cognitive Governance
- Iteratively refine responses based on validation feedback
- Compare initial vs final responses to see governance improvements

## The Value Proposition

**Before**: "Here are tools to check if your AI output is good"

**After**: "Give me a prompt, I'll generate a governed response guaranteed to meet standards"

## How It Works

1. User gives prompt
2. Claude Code generates response after consulting SIM-ONE docs via MCP
3. Five Laws Validator scores it + provides violations/recommendations
4. If score < 80%, Claude Code refines using that feedback
5. Repeat up to 3 times
6. Return guaranteed-quality response

## Requirements

- Claude Code CLI installed and authenticated
- SIM-ONE MCP connector configured
- Five Laws Validator (from SIM-ONE)

In [ ]:
# Standard library imports
import sys
import json
import subprocess
from pathlib import Path
from typing import Dict, Any, Optional, Literal

# Add SIM-ONE to Python path
SIMONE_ROOT = Path("../code").resolve()
sys.path.insert(0, str(SIMONE_ROOT))

# Import Five Laws validators
from mcp_server.protocols.governance.five_laws_validator.law1_architectural_intelligence import ArchitecturalIntelligenceProtocol
from mcp_server.protocols.governance.five_laws_validator.law2_cognitive_governance import CognitiveGovernanceProtocol
from mcp_server.protocols.governance.five_laws_validator.law3_truth_foundation import TruthFoundationProtocol
from mcp_server.protocols.governance.five_laws_validator.law4_energy_stewardship import EnergyStewardshipProtocol
from mcp_server.protocols.governance.five_laws_validator.law5_deterministic_reliability import DeterministicReliabilityProtocol

print("[+] Setup complete")

In [ ]:
class FiveLawsValidator:
    """Unified validator for all Five Laws of Cognitive Governance."""

    def __init__(self):
        self.law1 = ArchitecturalIntelligenceProtocol()
        self.law2 = CognitiveGovernanceProtocol()
        self.law3 = TruthFoundationProtocol()
        self.law4 = EnergyStewardshipProtocol()
        self.law5 = DeterministicReliabilityProtocol()

        self.thresholds = {
            "lenient": 60.0,
            "moderate": 70.0,
            "strict": 85.0
        }

    def validate(
        self,
        text: str,
        strictness: Literal["lenient", "moderate", "strict"] = "moderate",
        context: Optional[Dict[str, Any]] = None
    ) -> Dict[str, Any]:
        """Validate text against all Five Laws."""
        if not text or not text.strip():
            raise ValueError("Text to validate cannot be empty")

        validation_context = context or {}
        validation_context["text"] = text

        # Validate against each law
        law1_result = self.law1.evaluate(validation_context)
        law2_result = self.law2.evaluate(validation_context)
        law3_result = self.law3.evaluate(validation_context)
        law4_result = self.law4.evaluate(validation_context)
        law5_result = self.law5.evaluate(validation_context)

        # Extract scores
        law1_score = law1_result.get("score", 0.0)
        law2_score = law2_result.get("score", 0.0)
        law3_score = law3_result.get("score", 0.0)
        law4_score = law4_result.get("score", 0.0)
        law5_score = law5_result.get("score", 0.0)

        overall_compliance = (
            law1_score + law2_score + law3_score + law4_score + law5_score
        ) / 5.0

        # Collect violations and recommendations
        violations = []
        recommendations = []
        strengths = []

        for law_name, law_result in [
            ("Law 1", law1_result),
            ("Law 2", law2_result),
            ("Law 3", law3_result),
            ("Law 4", law4_result),
            ("Law 5", law5_result),
        ]:
            violations.extend(
                [f"{law_name}: {v}" for v in law_result.get("violations", [])]
            )
            recommendations.extend(
                [f"{law_name}: {r}" for r in law_result.get("recommendations", [])]
            )
            if law_result.get("score", 0) >= self.thresholds[strictness]:
                strengths.append(f"{law_name}: Strong compliance")

        threshold = self.thresholds[strictness]
        passed = overall_compliance >= threshold

        return {
            "scores": {
                "overall_compliance": overall_compliance,
                "law1_architectural_intelligence": law1_score,
                "law2_cognitive_governance": law2_score,
                "law3_truth_foundation": law3_score,
                "law4_energy_stewardship": law4_score,
                "law5_deterministic_reliability": law5_score,
            },
            "violations": violations,
            "recommendations": recommendations,
            "strengths": strengths,
            "pass_fail_status": "PASS" if passed else "FAIL",
        }


# Initialize validator
validator = FiveLawsValidator()
print("[+] Five Laws Validator initialized")

In [ ]:
def _clean_unicode(text_list: list) -> list:
    """Remove Unicode characters that cause Windows encoding errors."""
    cleaned = []
    for text in text_list:
        # The notebook already has ASCII, but clean any that slip through
        text = str(text)
        cleaned.append(text)
    return cleaned


def _generate_response(prompt: str, is_initial: bool = True) -> str:
    """
    Generate response using Claude Code CLI.
    
    Args:
        prompt: The prompt to send to Claude
        is_initial: Whether this is the initial generation (adds MCP instruction)
    
    Returns:
        Generated response text
    """
    # Add MCP connector instruction for initial generation
    if is_initial:
        full_prompt = f"""Before responding, use the SIM-ONE MCP connector to review the latest SIM-ONE Framework documentation, especially the Five Laws of Cognitive Governance.

User request: {prompt}

Generate a response that adheres to the SIM-ONE Framework's Five Laws."""
    else:
        # Refinement prompts already include MCP instruction
        full_prompt = prompt
    
    try:
        result = subprocess.run(
            ["claude", "-p", full_prompt],
            capture_output=True,
            text=True,
            timeout=120  # 2 minute timeout
        )
        
        if result.returncode != 0:
            raise RuntimeError(f"Claude Code CLI error: {result.stderr}")
        
        # Return the stdout as the response
        return result.stdout.strip()
    
    except subprocess.TimeoutExpired:
        raise RuntimeError("Claude Code CLI timed out after 2 minutes")
    except FileNotFoundError:
        raise RuntimeError("Claude Code CLI not found. Please install and authenticate Claude Code.")


def _validate_response(text: str, strictness: str) -> dict:
    """
    Validate response using Five Laws Validator.
    
    Args:
        text: Text to validate
        strictness: Validation strictness level
    
    Returns:
        Validation result with cleaned Unicode
    """
    result = validator.validate(text, strictness=strictness)
    
    # Clean Unicode from violations/recommendations before returning
    return {
        "score": result["scores"]["overall_compliance"],
        "violations": _clean_unicode(result["violations"]),
        "recommendations": _clean_unicode(result["recommendations"]),
        "strengths": _clean_unicode(result["strengths"]),
        "individual_scores": result["scores"]
    }


def _create_refinement_prompt(original_prompt: str, previous_response: str, 
                             validation_result: dict, threshold: float) -> str:
    """
    Create refinement prompt with MCP connector instruction.
    
    Args:
        original_prompt: The original user prompt
        previous_response: The response that failed validation
        validation_result: Validation results with violations/recommendations
        threshold: The threshold score to achieve
    
    Returns:
        Refinement prompt for Claude Code
    """
    # Identify which laws failed
    failed_laws = []
    for law_name, score in validation_result['individual_scores'].items():
        if score < threshold:
            # Convert law1_architectural_intelligence -> Law 1 Architectural Intelligence
            law_num = law_name.split('_')[0].replace('law', 'Law ')
            law_title = ' '.join(law_name.split('_')[1:]).title()
            failed_laws.append(f"{law_num} {law_title}")
    
    return f"""Before responding, use the SIM-ONE MCP connector to review the latest SIM-ONE Framework documentation, especially regarding:
{chr(10).join(f"- {law}" for law in failed_laws)}

Original user request: {original_prompt}

Previous response:
{previous_response}

Governance validation score: {validation_result['score']:.1f}%

Issues found:
{chr(10).join(f"- {v}" for v in validation_result['violations'][:5])}

Recommendations for improvement:
{chr(10).join(f"- {r}" for r in validation_result['recommendations'][:5])}

Please revise the response using the SIM-ONE Framework documentation to address these specific governance issues and improve the score above {threshold}%.
Focus on the specific violations and recommendations listed above."""


print("[+] Helper functions defined")

In [ ]:
def compose_governed_response(
    prompt: str,
    threshold: float = 80.0,
    max_iterations: int = 3,
    strictness: Literal["lenient", "moderate", "strict"] = "moderate"
) -> Dict[str, Any]:
    """
    Generate and iteratively refine AI response until it meets governance standards.
    
    Args:
        prompt: User's request/prompt
        threshold: Minimum governance score required (0-100)
        max_iterations: Maximum refinement attempts
        strictness: Validation strictness level
    
    Returns:
        Dictionary containing:
        - initial_response: First generated response with validation
        - final_response: Final refined response with validation
        - iterations: Number of refinement iterations performed
        - passed: Whether final response meets threshold
        - improvement: Percentage point improvement from initial to final
    """
    print(f"\n[+] Generating governed response for: \"{prompt[:50]}...\"")
    print(f"[+] Threshold: {threshold}% | Max iterations: {max_iterations} | Strictness: {strictness}\n")
    
    # Generate initial response (with MCP connector instruction)
    print("[1/3] Generating initial response...")
    initial_text = _generate_response(prompt, is_initial=True)
    print(f"      Generated {len(initial_text)} characters")
    
    # Validate initial response
    print("[2/3] Validating initial response...")
    initial_validation = _validate_response(initial_text, strictness)
    print(f"      Score: {initial_validation['score']:.1f}%")
    
    current_text = initial_text
    current_validation = initial_validation
    current_score = initial_validation["score"]
    
    # Iterate up to max_iterations if score < threshold
    iteration = 0
    print("[3/3] Refining response if needed...")
    
    for iteration in range(1, max_iterations + 1):
        if current_score >= threshold:
            print(f"      [PASS] Score {current_score:.1f}% meets threshold {threshold}%")
            break
        
        print(f"      [Iteration {iteration}] Score {current_score:.1f}% below threshold, refining...")
        
        # Create refinement prompt (includes MCP connector instruction)
        refinement_prompt = _create_refinement_prompt(
            prompt, current_text, current_validation, threshold
        )
        
        # Generate refined response
        current_text = _generate_response(refinement_prompt, is_initial=False)
        current_validation = _validate_response(current_text, strictness)
        current_score = current_validation["score"]
        print(f"      [Iteration {iteration}] New score: {current_score:.1f}%")
    
    # Return comparison
    improvement = current_score - initial_validation["score"]
    passed = current_score >= threshold
    
    print(f"\n[+] Complete! Final score: {current_score:.1f}% | Improvement: {improvement:+.1f}% | Status: {'PASS' if passed else 'FAIL'}\n")
    
    return {
        "initial_response": {
            "text": initial_text,
            "score": initial_validation["score"],
            "violations": initial_validation["violations"],
            "recommendations": initial_validation["recommendations"]
        },
        "final_response": {
            "text": current_text,
            "score": current_score,
            "violations": current_validation["violations"],
            "recommendations": current_validation["recommendations"]
        },
        "iterations": iteration,
        "passed": passed,
        "improvement": improvement
    }


print("[+] Main tool function defined")

## Example 1: Simple Prompt

This example shows a straightforward prompt that should generate a well-governed response.

In [ ]:
result1 = compose_governed_response(
    prompt="Explain how the Five Laws of Cognitive Governance work together",
    threshold=75.0,
    max_iterations=3,
    strictness="moderate"
)

print("=" * 80)
print("RESULTS:")
print("=" * 80)
print(f"Initial score: {result1['initial_response']['score']:.1f}%")
print(f"Final score:   {result1['final_response']['score']:.1f}%")
print(f"Improvement:   {result1['improvement']:+.1f}%")
print(f"Iterations:    {result1['iterations']}")
print(f"Status:        {'PASS' if result1['passed'] else 'FAIL'}")

## Example 2: Viewing Detailed Feedback

This example shows how to access detailed violations and recommendations.

In [ ]:
result2 = compose_governed_response(
    prompt="Design a multi-agent AI system",
    threshold=80.0,
    max_iterations=3,
    strictness="moderate"
)

print("=" * 80)
print("INITIAL RESPONSE FEEDBACK:")
print("=" * 80)
print(f"Score: {result2['initial_response']['score']:.1f}%\n")

if result2['initial_response']['violations']:
    print("Top 3 Violations:")
    for i, v in enumerate(result2['initial_response']['violations'][:3], 1):
        print(f"  {i}. {v}")

if result2['initial_response']['recommendations']:
    print("\nTop 3 Recommendations:")
    for i, r in enumerate(result2['initial_response']['recommendations'][:3], 1):
        print(f"  {i}. {r}")

print("\n" + "=" * 80)
print("FINAL RESPONSE:")
print("=" * 80)
print(f"Score: {result2['final_response']['score']:.1f}%")
print(f"Status: {'PASS - Guaranteed quality!' if result2['passed'] else 'FAIL'}")